# ___Correlated evolution between a continuous trait and a discrete categorical trait___
--------------------------------------

### ___[`OUwie::hOUwieStarterGuide`](https://thej022214.github.io/OUwie/articles/hOUwieStarterGuide.html)___
### ___[`OUwie`](https://thej022214.github.io/OUwie/index.html)___

In [1]:
print(R.version$version.string)

[1] "R version 4.5.2 (2025-10-31 ucrt)"


In [2]:
suppressPackageStartupMessages({
    library("ape")
    library("phytools")
    library("nlme")
    library("corHMM")
    library("geiger")
    library("mkcor")
    library("OUwie")
    library("reshape2")
    library("ggplot2")
})

packageVersion("OUwie") # make sure it is 2.16 

[1] '2.16'

## ___Model fitting to dummy data___
-----------------

In [16]:
# example model fitting

data(tworegime)
dat <- data.frame(sp = tree$tip.label, X = sample(c(0, 1, 2), length(tree$tip.label), replace = TRUE), Y = sample(c(0, 1), length(tree$tip.label), replace = TRUE), FS = rnorm(length(tree$tip.label), 10, 3))
head(dat)

,sp,X,Y,FS
,<chr>,<dbl>,<dbl>,<dbl>
1,t1,0,1,9.365061
2,t2,0,0,10.502899
3,t3,2,0,5.821208
4,t4,0,0,9.402193
5,t5,1,1,11.637713
6,t6,0,0,8.751180


In [17]:
p <- c(0.01670113, 0.39489947, 0.18619839, 1.67259459, 0.16817414)  # my fixed set of parameters
pp_oum <- OUwie::hOUwie(tree, trait, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 25, p = p)  # you likely won't use this p argument

Negative values detected... adding 50 to the trait mean for optimization purposes
Your phylogeny had node labels, these have been removed.
Calculating likelihood from a set of fixed parameters.
[1]  0.01670113  0.39489947  0.18619839 51.67259459 50.16817414


In [17]:
pp_oum


Fit
    lnLTot   lnLDisc   lnLCont     AIC     AICc      BIC nTaxa nPars
 -25.15545 -5.260483 -18.62633 60.3109 61.34539 71.10532    64     5

Legend
  1   2 
"1" "2" 

Regime Rate matrix
           (1)        (2)
(1)         NA 0.01670113
(2) 0.01670113         NA

OU Estimates
             (1)       (2)
alpha  0.3948995 0.3948995
sigma2 0.1861984 0.1861984
theta  1.6725946 0.1681741


Half-life (another way of reporting alpha)
    (1)     (2) 
1.75525 1.75525 

In [6]:
# fitting without p
model <- OUwie::hOUwie(tree, trait, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 25) 

Negative values detected... adding 50 to the trait mean for optimization purposes
Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


## ___Fitting the `hOUwie` model to our data___
-------------------------

In [31]:
fred_tree <- ape::multi2di(ape::read.tree("../data/chapter2/uphylomaker/fredv3subset_collab_trait_n_states.tre"))
data <- read.csv("../data/chapter2/FREDv3subset/FRED_subset_collab_states_n_species_avg_traits.csv", row.names = "binominal")

In [32]:
data <- data.frame(binominal = as.factor(gsub(rownames(data), pattern = ' ', replacement = '_')), rd = data$F00679, srl = data$F00727, myco = as.factor(data$F00645))
row_indices <- match(fred_tree$tip.label, data$binominal)
all(data$binominal[row_indices] == fred_tree$tip.label)
data <- data[row_indices, ]
all(data$binominal == fred_tree$tip.label) # cool

[1] TRUE

[1] TRUE

In [33]:
length(unique(data$binominal)) == length(data$binominal) # good, no duplicates

[1] TRUE

In [34]:
levels(data$myco) # 6 unique values for mycorrhizal states

[1] "AM"   "AMEM" "AMNM" "EM"   "ErM"  "NM"

In [35]:
head(data)

,binominal,rd,srl,myco
,<fct>,<dbl>,<dbl>,<fct>
39,Anaphalis_aureopunctata,0.136950,479.1550,AMNM
40,Anaphalis_hancockii,0.178700,319.8579,AM
311,Solidago_decurrens,0.248225,202.8837,AM
129,Doellingeria_scabra,0.211300,219.2008,AM
54,Aster_tataricus,0.197000,297.1100,AM
49,Artemisia_igniaria,0.167200,204.1529,AM


In [36]:
# OUwie::hOUwie expects the columns in the following order => species name, categorical trait, continuous trait

data_rd <- data[, c("binominal", "myco", "rd")] # for root diameter
data_srl <- data[, c("binominal", "myco", "srl")] # for specific root length

In [ ]:
# look up https://thej022214.github.io/OUwie/reference/hOUwie.html before specifying the discrete_model, continuous_model arguments to OUwie::hOUwie() 

# discrete_model - Either a user-supplied index of parameters to be optimized or one of "ARD", "SYM", or "ER". 
# ARD: all rates differ. SYM: rates between any two states do not differ. ER: all rates are equal.

# continuous_model - Either a user-supplied index matrix specifying the continuous model parameters to be estimated or one of "BM1", "BMV", "OU1", "OUA", "OUV", "OUM", "OUVA", "OUMV", "OUMA", "OUMVA" 
# (See also getOUParamStructure).

# null.model - A boolean indicating whether the model being run is a character-independent model with rate heterogeneity. Rate.cat must be greater than 1.

# nSim - The number of stochastic maps evaluated per iteration of the ML search.

In [28]:
OUwie::getOUParamStructure("OUMVA", nObsState = 6)

,(1),(2),(3),(4),(5),(6)
alpha,1,2,3,4,5,6
sigma2,7,8,9,10,11,12
theta,13,14,15,16,17,18


In [50]:
# BM1 and OU1 models expect no link between the discrete and continuous characters. I.e., all of the parameters are identical for different states of the categorical vector.
# BMV on the other hand suggests that the rates of evolution differs between different states of the categorical vector, as measure by sigma2.
# FUCK R AND ITS GROTESQUE SYNTAX

runtime_OUM <- Sys.time()
model_rd_OUM <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, discrete_model = "ER", continuous_model = "OUM", nSim = 25) # n_starts = 14, ncores = 14) => these args are NOT supported on Windows???
runtime_OUM <- Sys.time() - runtime_OUM

runtime_BM1 <- Sys.time()
model_rd_BM1 <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, discrete_model = "ER", continuous_model = "BM1", nSim = 25)
runtime_BM1 <- Sys.time() - runtime_BM1

runtime_OU1 <- Sys.time()
model_rd_OU1 <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, discrete_model = "ER", continuous_model = "OU1", nSim = 25)
runtime_OU1 <- Sys.time() - runtime_OU1

runtime_BMV <- Sys.time()
model_rd_BMV <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, discrete_model = "ER", continuous_model = "BMV", nSim = 25)
runtime_BMV <- Sys.time() - runtime_BMV

Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 1, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


In [53]:
# damn
runtime_BM1 # 2.611249 mins
runtime_BMV # 20.52963 mins
runtime_OU1 # 4.900693 mins
runtime_OUM # 24.8017 mins

Time difference of 2.611249 mins

Time difference of 20.52963 mins

Time difference of 4.900693 mins

Time difference of 24.8017 mins

In [54]:
# summary of model performances
models <- list(bm1 = model_rd_BM1, bmv = model_rd_BMV, ou1 = model_rd_OU1, oum = model_rd_OUM)
OUwie::getModelTable(models)

,np,lnLik,DiscLik,ContLik,BIC,dBIC,BICwt
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
bm1,3,-216.9088,-171.5257,-42.30647,451.3395,158.30858,4.204467e-35
bmv,8,-203.9080,-171.4943,-28.85857,454.5411,161.51017,8.481950e-36
ou1,4,-134.8342,-171.5320,39.79815,293.0309,0.00000,9.999602e-01
oum,9,-130.3638,-171.4534,44.27947,313.2934,20.26246,3.981482e-05


In [56]:
# allowing rate and optima variation

runtime_OUM_null <- Sys.time()
model_rd_OUM_null <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 6, discrete_model = "ER", continuous_model = "OUM", nSim = 25, null.model = TRUE)
runtime_OUM_null <- Sys.time() - runtime_OUM_null

runtime_BM1_null <- Sys.time()
model_rd_BM1_null <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 6, discrete_model = "ER", continuous_model = "BM1", nSim = 25, null.model = TRUE)
runtime_BM1_null <- Sys.time() - runtime_BM1_null

runtime_OU1_null <- Sys.time()
model_rd_OU1_null <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 6, discrete_model = "ER", continuous_model = "OU1", nSim = 25, null.model = TRUE)
runtime_OU1_null <- Sys.time() - runtime_OU1_null

runtime_BMV_null <- Sys.time()
model_rd_BMV_null <- OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 6, discrete_model = "ER", continuous_model = "BMV", nSim = 25, null.model = TRUE)
runtime_BMV_null <- Sys.time() - runtime_BMV_null

Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 6, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 6, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 6, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, rate.cat = 6, :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


In [57]:
runtime_BM1_null # 5.07681 mins
runtime_BMV_null # 52.09613 mins
runtime_OU1_null # 11.27017 mins
runtime_OUM_null # 41.80107 mins

Time difference of 5.07681 mins

Time difference of 52.09613 mins

Time difference of 11.27017 mins

Time difference of 41.80107 mins

In [59]:
models_null <- list(bm1 = model_rd_BM1_null, bmv = model_rd_BMV_null, ou1 = model_rd_OU1_null, oum = model_rd_OUM_null)
OUwie::getModelTable(models_null)

,np,lnLik,DiscLik,ContLik,BIC,dBIC,BICwt
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
bm1,38,-353.8592,-341.8731,-47.70111,929.6627,164.35894,2.028280e-36
bmv,43,-262.1252,-280.2260,-35.66062,775.3980,10.09420,6.386886e-03
ou1,39,-268.7594,-242.5410,-34.89080,765.3038,0.00000,9.936131e-01
oum,44,-283.6477,-293.9755,-15.41420,824.2837,58.97995,1.548404e-13


In [ ]:
avg_models_null <- OUwie::getModelAvgParams(models_null)
# avg_models_null

In [85]:
# options(repr.plot.width=20, repr.plot.height=12)

plot_df <- reshape2::melt(avg_models_null)
plot <- ggplot(plot_df, aes(x = tip_state, y = value, color = tip_state)) + 
    geom_point(size = 5, shape = 21) + 
    stat_summary(fun = mean, geom = "point", aes(group = 1, size = 2)) +
    stat_summary(fun.data = "mean_se", geom = "errorbar", aes(group = 1), width = 0.15, color = "black") +
    theme_classic(base_size = 22) + facet_wrap(~variable, scales = "free")

ggplot2::ggsave(plot = plot, filename = "../plots/hOUwie_stats.png", device = "png", width = 22, height = 12, units = "in", dpi = 750)

Using tip_state as id variables



In [58]:
# serialize all the models, we DO NOT WANT TO SEPND ANOTHER THREE HOURS, repeating this shite

saveRDS(object = model_rd_OUM, file = "../data/chapter2/hOUwie/houwie_rd_OUM.rds")
saveRDS(object = model_rd_BM1, file = "../data/chapter2/hOUwie/houwie_rd_BM1.rds")
saveRDS(object = model_rd_OU1, file = "../data/chapter2/hOUwie/houwie_rd_OU1.rds")
saveRDS(object = model_rd_BMV, file = "../data/chapter2/hOUwie/houwie_rd_BMV.rds")

saveRDS(object = model_rd_OUM_null, file = "../data/chapter2/hOUwie/houwie_rd_OUM_null.rds")
saveRDS(object = model_rd_BM1_null, file = "../data/chapter2/hOUwie/houwie_rd_BM1_null.rds")
saveRDS(object = model_rd_OU1_null, file = "../data/chapter2/hOUwie/houwie_rd_OU1_null.rds")
saveRDS(object = model_rd_BMV_null, file = "../data/chapter2/hOUwie/houwie_rd_BMV_null.rds")

In [17]:
best_fit <- readRDS("../data/chapter2/hOUwie/houwie_rd_BM1_null.rds") # works alright
# best_fit

In [45]:
# with 6 states, assuming each transition happens at a different rate, 6x6 transition rates will be possible?

all <- expand.grid(from = unique(data$myco), to = unique(data$myco)) 
perms <- all[apply(all, 1, function(x) {length(unique(x)) == 2}),]
perms

,from,to
,<fct>,<fct>
2,AM,AMNM
3,ErM,AMNM
4,AMEM,AMNM
5,NM,AMNM
6,EM,AMNM
7,AMNM,AM
9,ErM,AM
10,AMEM,AM
11,NM,AM


In [46]:
# try fitting an OUMVA model - allows variation in pull towards the optima (Alpha), regime continuous trait optioma (Mean) & rate of evolution (Variance)
# all while allowing character dependence
# while allowing all rates in discrete character shifts to differ i.e AM -> EcM if different to EcM -> AM
tm <- Sys.time()
oumva <- OUwie::hOUwie(phy = fred_tree, data = data_rd, discrete_model = "ARD", continuous_model = "OUMVA", null.model = TRUE, nSim = 25, rate.cat = 36)
tm <- Sys.time() - tm

Warning message:
"From feedback from a user, we have concerns about the likelihood function for the OUMA and OUMVA models. As of Sept. 4, 2025, we recommend not using these models for now, but we expect the situation to be resolved soon (either with corrected likelihood functions or a definite proof that they are ok as is). That said, we leave these as options for reproducibility and debugging. If you do use them, be sure to note the package version and report this in your work."
Warning message in OUwie::hOUwie(phy = fred_tree, data = data_rd, discrete_model = "ARD", :
"Your phylogeny edge lengths of 0. Adding 1e-5"


Your phylogeny had node labels, these have been removed.
Starting a thorough search with 25 simmaps using the nlopt_ln optimization protocol...


In [47]:
tm # 49.71021 mins

Time difference of 49.71021 mins

In [48]:
oumva # that's lame


Fit
    lnLTot   lnLDisc   lnLCont      AIC    AICc      BIC nTaxa nPars
 -1132.143 -1159.826 -37.55707 7160.286 1464.18 16562.18   344  2448

Legend
     1      2      3      4      5      6 
  "AM" "AMEM" "AMNM"   "EM"  "ErM"   "NM" 

Regime Rate matrix
              (1A)         (2A)         (3A)         (4A)         (5A)
(1A)            NA 0.0006071964 0.0006071964 0.0006071964 0.0006071964
(2A)  0.0006071964           NA 0.0006071964 0.0006071964 0.0006071964
(3A)  0.0003414517 0.0006071964           NA 0.0006071964 0.0006071964
(4A)  0.0006071964 0.0006071964 0.0006071964           NA 0.0006071964
(5A)  0.0006071964 0.0006071964 0.0006071964 0.0006071964           NA
(6A)  0.0006071964 0.0006071964 0.0006071964 0.0006071964 0.0006071964
(1B)  0.0006071964           NA           NA           NA           NA
(2B)            NA 0.0006071964           NA           NA           NA
(3B)            NA           NA 0.0006071964           NA           NA
(4B)            NA           NA  